# CU7 Performance Assessment (MOCK – practice) — PinjamPantas (starter)
Fill in your name and the last four digits of your IC, then work through the Tasks of the paper **in this notebook**.
First create your own data file in a terminal: `python make_my_data.py <L4>`  (the file `loan_applicants.csv` appears in this folder).
Keep the notebook runnable from top to bottom.

In [3]:
import warnings; warnings.filterwarnings("ignore")
import mlflow, numpy as np, pandas as pd

NAME = "PETER LING HUA ZHONG"      # e.g. "AISYAH BINTI AHMAD"
L4   = "0161"                # last four digits of your IC, keep the leading zero

# ---- your personal values (rules in the paper) ----
SEED      = int(L4)
TEST_SIZE = 0.20 if int(L4[3]) % 2 == 0 else 0.25
DEPTH     = 3 + int(L4[1]) % 4
C1        = round(0.04 * (int(L4[2]) + 1), 2)
PORT      = 5100 + int(L4[2:])
EXP       = f"PinjamPantas_Default_{L4}"
REG       = f"pinjampantas_default_{L4}"
print(dict(SEED=SEED, TEST_SIZE=TEST_SIZE, DEPTH=DEPTH, C1=C1, PORT=PORT, EXP=EXP, REG=REG))

{'SEED': 161, 'TEST_SIZE': 0.25, 'DEPTH': 4, 'C1': 0.28, 'PORT': 5161, 'EXP': 'PinjamPantas_Default_0161', 'REG': 'pinjampantas_default_0161'}


## Task 1 — Set-up, split and pipeline

In [7]:
# your code
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


mlflow.set_tracking_uri(f"sqlite:///pinjampantas_{L4}.db")
mlflow.set_experiment(EXP)
print("Tracking URI:", mlflow.get_tracking_uri())

df = pd.read_csv("loan_applicants.csv")
print(df.shape); display(df.head())


X = df.drop(columns=["applicant_id", "defaulted"])
y = df["defaulted"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print("train:", X_train.shape, " test:", X_test.shape)

cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = df.select_dtypes(include="number").columns.tolist()
print("categorical columns:", cat_cols)
print("numerical:", num_cols)

def make_pipeline(model):
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
                            remainder="passthrough")
    return Pipeline([("prep", pre), ("model", model)])

Tracking URI: sqlite:///pinjampantas_0161.db
(1200, 10)


,applicant_id,loan_amount_rm,loan_purpose,age,late_payments,employment,region,housing,monthly_income_rm,defaulted
0,A0001,11804.0,Home,36,0,Self-employed,South,Rent,7708.0,0
1,A0003,10015.0,Education,28,0,Contract,North,Own,6258.0,0
2,A0005,13875.0,Home,38,1,Self-employed,South,Own,1500.0,0
3,A0006,45661.0,Education,32,2,Contract,Central,Family,3454.0,1
4,A0007,10810.0,Personal,21,0,Salaried,Central,Own,2530.0,0


train: (900, 8)  test: (300, 8)
categorical columns: ['loan_purpose', 'employment', 'region', 'housing']
numerical: ['loan_amount_rm', 'age', 'late_payments', 'monthly_income_rm', 'defaulted']


## Task 2 — Track the candidates in ONE experiment

In [10]:
# your code
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def get_metrics(y_true,y_pred):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true,y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred))
    }

candidates = [
    ("linear-base", LinearRegression(), {}, "linear"),
    (f"ridge-a{C1:g}", Ridge(alpha=C1), {"alpha":C1}, "linear"),
    (f"rf-d{DEPTH}", RandomForestRegressor(n_estimators=100,max_depth=DEPTH,random_state=SEED),{"n_estimators":100, "max_depth":DEPTH}, "tree_ensemble"),
]

run_ids = {}
for prefix,model,hparams,mtype in candidates:
    run_name = f"{prefix}-{L4}"
    with mlflow.start_run(run_name=run_name) as run:
        pipe = make_pipeline(model).fit(X_train,y_train)
        train_m = get_metrics(y_train, pipe.predict(X_train))
        test_m = get_metrics(y_test, pipe.predict(X_test))

        mlflow.log_params({"test_size":TEST_SIZE,"split_seed":SEED,"model":type(model).__name__, **hparams})
        mlflow.set_tags({"model_type":mtype,"candidate":NAME})
        mlflow.log_metrics({f"train_{k}": v for k, v in train_m.items()})
        mlflow.log_metrics({f"test_{k}": v for k, v in test_m.items()})

        sig = infer_signature(X_train, pipe.predict(X_train))
        mlflow.sklearn.log_model(pipe,name="model",signature=sig,input_example=X_train.head(3),
                                skops_trusted_types=["sklearn.tree._tree.Tree"])

        run_ids[run_name] = run.info.run_id
        print(f"{run_name:<22} train={train_m}\n{'':22} test={test_m}")

linear-base-0161       train={'rmse': 0.3378517911589602, 'mae': 0.2595958532668929, 'r2': 0.35980373386225295}
                       test={'rmse': 0.34821400312322165, 'mae': 0.26642437742835157, 'r2': 0.31534166024224153}


ridge-a0.28-0161       train={'rmse': 0.3378519640778836, 'mae': 0.2596507467266999, 'r2': 0.3598030785330206}
                       test={'rmse': 0.34819486283718976, 'mae': 0.2664667769334972, 'r2': 0.31541692543077704}


rf-d4-0161             train={'rmse': 0.30672389077007267, 'mae': 0.21747736485028524, 'r2': 0.47233790853698954}
                       test={'rmse': 0.3460403034870799, 'mae': 0.2526285846544006, 'r2': 0.32386283660400694}


## Task 3 — Choose the best run from code and justify it

In [12]:
# your code
runs = mlflow.search_runs(
    experiment_names=[EXP],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.test_rmse ASC"])

runs["gap_rmse"] = (runs["metrics.train_rmse"] - runs["metrics.test_rmse"]).abs()
display(runs[["tags.mlflow.runName", "tags.model_type", "metrics.test_rmse", "metrics.test_mae", "metrics.test_r2", "metrics.train_rmse","gap_rmse"]])

best = runs.iloc[0]
BEST_RUN_ID = best["run_id"]
BEST_NAME = best["tags.mlflow.runName"]
BASE_NAME = candidates[0][0] + f"-{L4}"
BASE_RUN_ID = run_ids[BASE_NAME]
print(f"Best run: {BEST_NAME} (test_rmse = {best['metrics.test_rmse']:.4f})")

,tags.mlflow.runName,tags.model_type,metrics.test_rmse,metrics.test_mae,metrics.test_r2,metrics.train_rmse,gap_rmse
0,rf-d4-0161,tree_ensemble,0.346040,0.252629,0.323863,0.306724,0.039316
1,ridge-a0.28-0161,linear,0.348195,0.266467,0.315417,0.337852,0.010343
2,linear-base-0161,linear,0.348214,0.266424,0.315342,0.337852,0.010362
3,rf-d4-0161,tree_ensemble,NaN,NaN,NaN,NaN,NaN
4,logreg-c0.28-0161,linear,NaN,NaN,NaN,NaN,NaN
5,logreg-base-0161,linear,NaN,NaN,NaN,NaN,NaN


Best run: rf-d4-0161 (test_rmse = 0.3460)


**Why I chose this run (2–3 sentences):I chose logreg-c0.28-0161 because it has the highest test_f1 (0.651), higher than the baseline logreg-base-0161 (0.625) and the random forest rf-d4-0161 (0.442). F1 is the right metric for loan defaults because it balances catching real defaulters (recall 0.594) with not rejecting good applicants (precision 0.719). Its train F1 (0.640) is close to its test F1, so the model is not overfitting and should work on new applicants.

## Task 4 — Register, alias and roll back

In [13]:
# your code
from mlflow import MlflowClient
client = MlflowClient()

v1 = mlflow.register_model(f"runs:/{BASE_RUN_ID}/model",REG)
client.set_registered_model_alias(REG, "champion",v1.version)
print("v1 =", BASE_NAME, "-> champion")

v2 = mlflow.register_model(f"runs:/{BEST_RUN_ID}/model", REG)
client.set_registered_model_alias(REG, "champion", v2.version)
client.set_model_version_tag(REG, v2.version, "approved_by", NAME)
client.update_model_version(REG, v2.version,
                           description=f"{BEST_NAME}: best test_rmse = {best['metrics.test_rmse']:.4f} on the held-out test set "
                           f"(split seed {SEED}, test_size {TEST_SIZE}). Promoted over baseline v{v1.version}.")
print("v2 =",BEST_NAME, "-> champion")

client.set_registered_model_alias(REG, "champion", v1.version)
print("ROLLBACK champion ->", client.get_model_version_by_alias(REG, "champion").version)

client.set_registered_model_alias(REG, "champion", v2.version)
mv = client.get_model_version_by_alias(REG, "champion")
print(f"FINAL champion -> v{mv.version} | run {mv.run_id} | tags {mv.tags}")

Successfully registered model 'pinjampantas_default_0161'.
2026/10/18 21:24:51 WARNING mlflow.tracking._model_registry.fluent: Run with id 6d2e77dada0442c38d155ca2bdc38517 has no artifacts at artifact path 'model', registering model based on models:/m-03793f7675394df8bd99ebfa2da6cbf1 instead


Created version '1' of model 'pinjampantas_default_0161'.
Registered model 'pinjampantas_default_0161' already exists. Creating a new version of this model...


2026/10/18 21:24:51 WARNING mlflow.tracking._model_registry.fluent: Run with id d4ad6b128dc84372b227603aef201952 has no artifacts at artifact path 'model', registering model based on models:/m-ef4a2db1d8ab406ea404f302cb8f0d06 instead


Created version '2' of model 'pinjampantas_default_0161'.

v1 = linear-base-0161 -> champion


v2 = rf-d4-0161 -> champion
ROLLBACK champion -> 1
FINAL champion -> v2 | run d4ad6b128dc84372b227603aef201952 | tags {'approved_by': 'PETER LING HUA ZHONG'}


## Task 5 — Serve the champion and call it

In [16]:
# your code
import os, sys, time, subprocess, requests

MODEL_URI = f"models:/{REG}@champion"
URL = f"http://127.0.0.1:{PORT}"

def server_up():
    try:    return requests.get(URL + "/ping", timeout=2).status_code == 200
    except requests.exceptions.RequestException: return False

server = None
if not server_up():
    env = {**os.environ, "MLFLOW_TRACKING_URI": f"sqlite:///pinjampantas_{L4}.db"}
    server = subprocess.Popen([sys.executable, "-m", "mlflow", "models", "serve", "-m", MODEL_URI,
                               "-p", str(PORT), "--env-manager", "local"], env=env)
    for _ in range(90):
        if server_up(): break
        time.sleep(1)
print("server up:", server_up(), "on", URL)

server up: True on http://127.0.0.1:5161


In [17]:
row = X.iloc[[int(L4) % len(X)]]
display(row)

payload = {"dataframe_split": row.to_dict(orient="split", index=False)}
resp = requests.post(URL + "/invocations", json=payload, timeout=30)
print("HTTP", resp.status_code, resp.json())

pred = resp.json()["predictions"][0]
local = mlflow.pyfunc.load_model(MODEL_URI).predict(row)[0]
print(f"server predicted price (RM): {pred}   | same model loaded locally: {local}")

,loan_amount_rm,loan_purpose,age,late_payments,employment,region,housing,monthly_income_rm
161,9184.0,Education,37,2,Salaried,Central,Rent,4459.0


HTTP 200 {'predictions': [0.1347356112820782]}


server predicted price (RM): 0.1347356112820782   | same model loaded locally: 0.1347356112820782


In [20]:
if server is not None: server.terminate()